# Module T5-01: LLM Fine-tuning

**Tier 5 — Modern AI for Science | Module 01**

*Prerequisites: Tier 3 Module 10 (Deep Learning for Biology), PyTorch basics.*

**GPU required** for the training cells. Theory and pattern cells run on CPU. Run on free-tier Colab (T4 GPU).

---

This module covers fine-tuning large language models for domain-specific instruction following using LoRA, quantization, and the `trl` SFTTrainer.

**By the end you will be able to:**
- Explain LoRA: low-rank adapter math and parameter counts
- Apply 4-bit NF4 quantization with bitsandbytes
- Format instruction datasets using chat templates
- Run SFTTrainer to fine-tune a base model
- Generate synthetic instruction data for domain adaptation

**Attribution:** *Patterns inspired by Unsloth AI and Manuel Faysse fine-tuning tutorials. Uses public instruction datasets from HuggingFace Hub.*

---

[← Previous: Sequence Alignment with Dynamic Programming](../../Tier_4_Algorithms_and_Data_Structures/10_Dynamic_Programming/04_sequence_alignment.ipynb) | [Next: LLM Training Systems →](./02_llm_training_systems.ipynb)

---

## 1. LoRA: Low-Rank Adaptation

Full fine-tuning updates all W ≈ 7B parameters — prohibitive without many A100s. LoRA instead trains two small matrices A and B (rank r) that decompose the weight update:

$$\Delta W = B \cdot A, \quad B \in \mathbb{R}^{d \times r}, \; A \in \mathbb{R}^{r \times k}$$

During inference: W_eff = W_pretrained + (α/r) · B·A

**Parameter savings:**
| Model | Full FT params | LoRA (r=16) params | Reduction |
|---|---|---|---|
| 7B (Mistral, Llama) | 7,000,000,000 | ~6,800,000 | 1000× |
| 13B | 13,000,000,000 | ~12,600,000 | 1000× |

**Which modules to target?**
- Minimum: `q_proj`, `v_proj` (query and value in attention)
- Recommended: + `k_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`

In [ ]:
def lora_param_count(model_dim, n_heads, n_layers, rank, target_modules):
    """
    Estimate LoRA parameter count.
    model_dim: hidden size (e.g. 4096 for 7B models)
    n_heads: number of attention heads
    n_layers: number of transformer layers
    """
    params_per_layer = 0
    if "q_proj" in target_modules:
        params_per_layer += 2 * rank * model_dim  # A and B matrices
    if "v_proj" in target_modules:
        params_per_layer += 2 * rank * model_dim
    if "k_proj" in target_modules:
        params_per_layer += 2 * rank * model_dim
    if "o_proj" in target_modules:
        params_per_layer += 2 * rank * model_dim

    total = params_per_layer * n_layers
    return total

# Mistral-7B architecture
total_params = 7_000_000_000
for rank in [8, 16, 32, 64]:
    lora_p = lora_param_count(4096, 32, 32, rank,
                               ["q_proj","v_proj","k_proj","o_proj"])
    pct = lora_p / total_params * 100
    print(f"r={rank:3d}: {lora_p:>12,} trainable params ({pct:.3f}% of 7B)")

## 2. Quantization: 4-bit NF4

Quantization reduces the numerical precision of model weights from float32 (4 bytes/param) to 4-bit (0.5 bytes/param) — an 8× memory reduction.

**NF4 (NormalFloat 4-bit):** Designed for normally distributed weights (which transformer weights typically are). Values are mapped to 16 fixed points optimally placed for a normal distribution.

| Format | Bits | Memory for 7B | Quality |
|---|---|---|---|
| float32 | 32 | 28 GB | Reference |
| bfloat16 | 16 | 14 GB | ~same as fp32 |
| INT8 | 8 | 7 GB | Small quality loss |
| NF4 | 4 | 3.5 GB | Small quality loss |

**QLoRA:** Combine 4-bit base model (frozen) + LoRA adapters (bf16, trainable). Train only LoRA; inference dequantizes NF4 on the fly.

In [ ]:
# Quantization configuration (runs on CPU; actual loading requires GPU)
try:
    from transformers import BitsAndBytesConfig
    import torch

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",          # NormalFloat 4-bit
        bnb_4bit_compute_dtype=torch.bfloat16,  # compute in bf16
        bnb_4bit_use_double_quant=True,     # double quantization for extra savings
    )
    print("BitsAndBytesConfig created successfully")
    print(f"  quant_type: {bnb_config.bnb_4bit_quant_type}")
    print(f"  compute_dtype: {bnb_config.bnb_4bit_compute_dtype}")
    print(f"  double_quant: {bnb_config.bnb_4bit_use_double_quant}")
except ImportError:
    print("bitsandbytes not installed — pattern shown below")
    print("""
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    quantization_config=bnb_config,
    device_map="auto",
)
""")

## 3. Chat Templates

Instruction-tuned models expect inputs in a specific format. The format varies by model family:

**Mistral / Llama-2 Instruct:**
```
[INST] <<SYS>>
{system}
<</SYS>>

{user} [/INST] {assistant}</s>
```

**Alpaca:**
```
### Instruction:
{instruction}

### Input:
{context}

### Response:
{output}
```

**ChatML (OpenAI-compatible):**
```
<|im_start|>system
{system}<|im_end|>
<|im_start|>user
{user}<|im_end|>
<|im_start|>assistant
{assistant}<|im_end|>
```

In [ ]:
def format_mistral(system, user, assistant=None):
    prompt = f"[INST] <<SYS>>\n{system}\n<</SYS>>\n\n{user} [/INST]"
    if assistant:
        prompt += f" {assistant}</s>"
    return prompt

def format_alpaca(instruction, input_text="", output=""):
    prompt = f"### Instruction:\n{instruction}\n"
    if input_text:
        prompt += f"\n### Input:\n{input_text}\n"
    prompt += f"\n### Response:\n{output}"
    return prompt

def format_chatml(system, user, assistant=None):
    prompt = f"<|im_start|>system\n{system}<|im_end|>\n"
    prompt += f"<|im_start|>user\n{user}<|im_end|>\n"
    prompt += "<|im_start|>assistant\n"
    if assistant:
        prompt += f"{assistant}<|im_end|>"
    return prompt

# Examples
sys = "You are a helpful bioinformatics assistant."
usr = "What does RPKM stand for and how does it differ from TPM?"
asst = "RPKM (Reads Per Kilobase per Million) normalizes by library size and gene length. TPM also normalizes by both, but divides by the sum of all RPK values, making samples directly comparable."

print("=== Mistral format ===")
print(format_mistral(sys, usr, asst))
print("\n=== Alpaca format ===")
print(format_alpaca(usr, output=asst))

## 4. Dataset Preparation

A supervised fine-tuning (SFT) dataset consists of (instruction, response) pairs. Sources:

| Source type | Example | Size needed |
|---|---|---|
| Public instruction sets | Alpaca, OpenHermes | General capability |
| Domain-specific | Curated biomedical Q&A | Domain adaptation |
| Synthetic | Generated by a strong LLM | Data augmentation |

**Golden rules:**
1. Quality > quantity — 500 high-quality examples outperform 10k noisy ones
2. Diversity — cover all task types you want the model to handle
3. Format consistency — use the same template throughout
4. No contamination — hold out test examples before generating training data

In [ ]:
import pandas as pd

# Synthetic bioinformatics instruction dataset
bio_qa = [
    {
        "instruction": "Explain the difference between FASTQ and FASTA file formats.",
        "response": "FASTA stores sequences with a header line starting with '>'. FASTQ additionally stores per-base quality scores in Phred format, encoded as ASCII characters. FASTQ is the standard output format for sequencing platforms."
    },
    {
        "instruction": "What is the purpose of PCR duplicate marking in NGS pipelines?",
        "response": "PCR duplicates arise when the same DNA fragment is amplified and sequenced multiple times. Marking them prevents double-counting in variant calling and expression quantification, which would artificially inflate confidence in erroneous calls."
    },
    {
        "instruction": "Describe the three ontologies in Gene Ontology (GO).",
        "response": "Gene Ontology has three orthogonal ontologies: (1) Molecular Function (MF) — the biochemical activity of a gene product; (2) Biological Process (BP) — the pathway or larger process the gene product contributes to; (3) Cellular Component (CC) — where in the cell the gene product is active."
    },
    {
        "instruction": "What is the Bonferroni correction and when should you use it?",
        "response": "The Bonferroni correction divides the significance threshold α by the number of tests m, setting the per-test threshold to α/m. Use it when all tests are independent and you want strict family-wise error rate control. For correlated tests (e.g. genomic variants in LD), the Benjamini-Hochberg FDR procedure is usually preferred."
    },
    {
        "instruction": "Explain what a p-value is and what it is NOT.",
        "response": "A p-value is the probability of observing a test statistic at least as extreme as the one computed, assuming the null hypothesis is true. It is NOT: the probability that H0 is true, the probability that results occurred by chance, or a measure of effect size. Small p-values indicate the data are unlikely under H0, not that the effect is biologically meaningful."
    },
]

# Format as Alpaca
records = []
for qa in bio_qa:
    text = format_alpaca(qa["instruction"], output=qa["response"])
    records.append({"instruction": qa["instruction"], "response": qa["response"], "text": text})

df = pd.DataFrame(records)
print(f"Dataset: {len(df)} examples")
print(f"\nSample formatted text:\n{'='*60}")
print(df["text"].iloc[0])
print(f"{'='*60}")
print(f"\nAverage tokens (approx): {df['text'].str.len().mean() / 4:.0f}")

## 5. SFTTrainer Workflow

The `trl` library's `SFTTrainer` wraps HuggingFace `Trainer` with instruction-tuning conveniences: automatic packing, dataset text field selection, and LoRA integration.

In [ ]:
# Full SFTTrainer pattern (requires GPU; shown as executable template)
SFTTRAINER_PATTERN = '''
# ━━━━━ Install (Colab) ━━━━━
# !pip install -q unsloth trl peft bitsandbytes transformers accelerate

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
from datasets import Dataset
import torch

MODEL_NAME = "mistralai/Mistral-7B-v0.1"

# 1. Load quantized model
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config,
                                              device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# 2. Apply LoRA
lora_cfg = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj","v_proj","k_proj","o_proj"],
                       lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# 3. Prepare dataset
dataset = Dataset.from_list(records)  # records has "text" field

# 4. Train
training_args = TrainingArguments(
    output_dir="./results", num_train_epochs=2,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    learning_rate=2e-4, fp16=True, logging_steps=10,
    gradient_checkpointing=True,
)
trainer = SFTTrainer(model=model, args=training_args, train_dataset=dataset,
                      dataset_text_field="text", max_seq_length=2048, tokenizer=tokenizer)
trainer.train()
trainer.model.save_pretrained("./bio_assistant")
'''

print("SFTTrainer pattern:")
print(SFTTRAINER_PATTERN)

## 6. Synthetic Data Generation

When labelled examples are scarce, a stronger model (teacher) can generate synthetic training data for a smaller model (student) — this is called *knowledge distillation via instruction generation*.

**Pipeline:**
1. Provide the teacher with a context (paper abstract, textbook passage)
2. Ask it to generate diverse question-answer pairs
3. Filter by quality (length, format, relevance)
4. Fine-tune the student model on the synthetic pairs

In [ ]:
# Synthetic instruction generation pipeline (template)
def create_synthetic_instructions(context, n_pairs=5):
    """Generate synthetic Q&A from a context string."""
    meta_prompt = f"""Generate {n_pairs} diverse question-answer pairs about the following text.
Each pair should test a different aspect (definitions, mechanisms, comparisons, applications).
Format each pair as:
Q: <question>
A: <answer>

Text:
{context}

Pairs:"""
    # In practice, send meta_prompt to a strong LLM (GPT-4, Claude, Llama-3-70B)
    # Here we show the structure
    return meta_prompt

# Example context: abstract of a public paper
context = """
RNA-seq (RNA sequencing) is a transcriptomics technique that uses next-generation sequencing
to reveal the presence and quantity of RNA molecules in a biological sample. It provides a
snapshot of gene expression at a given moment. The pipeline involves: (1) library preparation
from total RNA or mRNA, (2) sequencing on Illumina or other platforms, (3) alignment to a
reference genome, (4) transcript quantification using tools like featureCounts or salmon,
and (5) differential expression analysis with DESeq2 or edgeR.
"""

print("Meta-prompt for synthetic instruction generation:")
print("="*60)
print(create_synthetic_instructions(context, n_pairs=3))
print("="*60)
print("\nThe above prompt would be sent to a teacher LLM.")
print("The resulting Q&A pairs are filtered and added to the training set.")

## 7. Evaluation

Standard NLP metrics (ROUGE, BLEU) correlate poorly with instruction-following quality. Better evaluation approaches:

| Method | Description | Cost |
|---|---|---|
| **Task-specific** | Accuracy on benchmark (MMLU, BioASQ) | Medium |
| **LLM-as-judge** | GPT-4 rates response on rubric (1–5) | Low |
| **Human evaluation** | Pairwise preference | High |
| **Perplexity** | Loss on held-out set | Trivial |

For bioinformatics models, evaluate on domain benchmarks: BioASQ, PubMedQA, MedMCQA.

In [ ]:
# Simple generation evaluation (CPU-friendly, using small model)
try:
    from transformers import pipeline
    # Use a tiny model for demonstration
    generator = pipeline("text-generation", model="gpt2", device=-1, max_new_tokens=50)
    test_prompts = [
        "What is a p-value?",
        "Explain FASTQ format.",
    ]
    for prompt in test_prompts:
        result = generator(f"### Question: {prompt}\n### Answer:", max_new_tokens=50, do_sample=False)
        print(f"Q: {prompt}")
        print(f"A: {result[0]['generated_text'][-100:]}")
        print()
except Exception as e:
    print(f"Pipeline demo: {e}")
    print("In practice: load your fine-tuned model and evaluate on held-out examples")

In [ ]:
# Exercise T5-01
# 1. Expand the bio_qa dataset with 10 more bioinformatics Q&A pairs.
#    Cover topics from Tiers 1–3 not yet in the dataset (e.g. BLAST, phylogenetics, RNA-seq).
# 2. Implement a simple quality filter for the dataset:
#    remove examples where response length < 30 words OR > 300 words.
#    How does this affect dataset size?
# 3. Compare the Mistral, Alpaca, and ChatML templates for the same Q&A pair.
#    Count tokens for each format. Which is most token-efficient?
# 4. Challenge: for rank r in [4, 8, 16, 32, 64], compute the LoRA parameter count
#    for a 7B model with target_modules = ["q_proj","v_proj","k_proj","o_proj","gate_proj","up_proj","down_proj"].
#    Plot trainable parameters vs r. At what r do you get ~1% of total parameters?

---

[← Previous: Sequence Alignment with Dynamic Programming](../../Tier_4_Algorithms_and_Data_Structures/10_Dynamic_Programming/04_sequence_alignment.ipynb) | [Next: LLM Training Systems →](./02_llm_training_systems.ipynb)

---